# ISG RAG Projesi - Uçtan Uca Pipeline ve Hybrid Retrieval Demo

Bu notebook, **SLAI Team 14** projesinde Evrim'in **Hybrid Retrieval (BM25 + Dense + RRF)** modülünü çalıştırmak için hazırlanmıştır.

> **GPU Kontrolü:** Üst menüden `Çalışma zamanı (Runtime)` -> `Çalışma zamanı türünü değiştir (Change runtime type)` -> **T4 GPU** seçildiğinden emin olun.

## 1. Adım: Depoyu Klonlama ve Kurulum

In [ ]:
!git clone https://github.com/yusufdinc974/SLAI.git
%cd SLAI
!pip install -r requirements.txt python-docx

## 2. Adım: Belgeleri İşleme ve Chunking (Mevzuat Parçalama)

In [ ]:
# Word dosyalarını JSON'a dönüştürür
!python -m document_processing.main

# Madde/fıkra hiyerarşisini koruyarak 5 farklı deney boyutu için chunk'ları üretir
!python -m document_processing.chunking.chunker

## 3. Adım: BGE-M3 Dense Embedding Üretimi (GPU) ve Qdrant İndeksleme

In [ ]:
# GPU desteğini garantiye almak için kod güncellemesi
with open("embeddings/generate_embeddings.py", "r", encoding="utf-8") as f:
    code = f.read()
code = code.replace('device="cpu"', 'device=("cuda" if torch.cuda.is_available() else "cpu")')
with open("embeddings/generate_embeddings.py", "w", encoding="utf-8") as f:
    f.write(code)

print("GPU ayarı yapıldı. Embedding'ler üretiliyor...")
!python -m embeddings.generate_embeddings

print("Embedding'ler Qdrant vektör veritabanına indeksleniyor...")
!python -m vectorstore.index_all_experiments

## 4. Adım: Evrim'in Hybrid Retrieval Sistemi (BM25 vs Dense vs RRF Karşılaştırması)

In [ ]:
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient

from retrieval import BM25Index, reciprocal_rank_fusion, RetrievalResult

print("1. BGE-M3 modeli ve Qdrant veritabanı yükleniyor...")
client = QdrantClient(path="data/qdrant")
model = SentenceTransformer("BAAI/bge-m3", device="cuda")

print("2. Mevzuat pasajları yükleniyor...")
passages = {}
chunks_dir = Path("data/chunks/size_1000_overlap_0")
for f in sorted(chunks_dir.glob("*.json")):
    with open(f, "r", encoding="utf-8") as fp:
        for chunk in json.load(fp):
            passages[chunk["passage_id"]] = chunk

print(f"-> Toplam {len(passages)} mevzuat pasajı belleğe alındı.")

# 3. Evrim'in BM25 İndeksi
bm25_corpus = {pid: chunk["text"] for pid, chunk in passages.items()}
bm25 = BM25Index(bm25_corpus)

# 4. Dense Arama Fonksiyonu
def search_dense(query: str, top_k: int = 20):
    query_vec = model.encode(query, normalize_embeddings=True, show_progress_bar=False).tolist()
    hits = client.search(collection_name="isg_size_1000_overlap_0", query_vector=query_vec, limit=top_k)
    return [RetrievalResult(h.payload["passage_id"], float(h.score)) for h in hits]

# 5. Hybrid Arama Fonksiyonu (RRF)
def search_hybrid(query: str, top_k: int = 3, candidate_top_k: int = 20):
    bm25_res = bm25.search(query, top_k=candidate_top_k)
    dense_res = search_dense(query, top_k=candidate_top_k)
    hybrid_res = reciprocal_rank_fusion([bm25_res, dense_res], top_k=top_k)
    return {"bm25": bm25_res[:top_k], "dense": dense_res[:top_k], "hybrid": hybrid_res}

def print_results(method_title, results):
    print(f"\n--- {method_title} ---")
    for rank, res in enumerate(results, 1):
        info = passages.get(res.passage_id, {})
        title = info.get("title", "")
        marker = info.get("section_marker") or ""
        text_preview = info.get("text", "").replace("\n", " ")[:130]
        print(f" {rank}. [{res.passage_id}] Skor: {res.score:.4f} | {title} {marker}")
        print(f"    Özet: {text_preview}...")

# --- TEST VE KARŞILAŞTIRMA ---
soru = "İş sağlığı ve güvenliği kurulu kaç çalışan olan işyerlerinde kurulmalıdır?"
print("=" * 75)
print(f"SORU: {soru}")
print("=" * 75)

karsilastirma = search_hybrid(soru, top_k=3)
print_results("1. BM25 (Kelime Bazlı / Lexical)", karsilastirma["bm25"])
print_results("2. Dense (BGE-M3 Vektörel / Qdrant)", karsilastirma["dense"])
print_results("3. Hybrid (Evrim'in RRF Füzyonu)", karsilastirma["hybrid"])